# Day 54 — Monitoring & model governance
Objectives:
- Monitor data quality and drift.
- Track model performance post-deployment.
- Governance basics: model registry, approvals, audit trail.
Note: This notebook simulates drift and basic monitoring metrics.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(42)
# Simulate baseline score distribution
y_true = rng.integers(0,2, size=5000)
y_score = y_true*0.7 + (1-y_true)*0.3 + rng.normal(0,0.1, size=5000)
baseline_auc = roc_auc_score(y_true, y_score)
baseline_auc


In [ ]:
# Simulate drift: feature shift lowers separability
y_true2 = rng.integers(0,2, size=5000)
y_score2 = y_true2*0.6 + (1-y_true2)*0.4 + rng.normal(0,0.15, size=5000)
live_auc = roc_auc_score(y_true2, y_score2)
delta = live_auc - baseline_auc
baseline_auc, live_auc, delta


## Data drift detection (simple)
- Monitor feature means/stds, PSI (Population Stability Index) for scoring bands.
Below is a simple PSI demo for score bins.

In [ ]:
def psi(expected, actual, bins=10):
    e_perc, _ = np.histogram(expected, bins=bins, range=(expected.min(), expected.max()))
    a_perc, _ = np.histogram(actual,   bins=bins, range=(expected.min(), expected.max()))
    e_perc = e_perc / e_perc.sum(); a_perc = a_perc / a_perc.sum()
    # add tiny value to avoid div/0
    e_perc = np.clip(e_perc, 1e-6, None); a_perc = np.clip(a_perc, 1e-6, None)
    return np.sum((a_perc - e_perc) * np.log(a_perc / e_perc))
psi_value = psi(y_score, y_score2)
psi_value


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — monitoring layers, drift signals, delayed labels, and governance decisions

### Mental model

Production monitoring has several layers. Data-quality checks ask
whether inputs are valid. Data drift compares input distributions.
Performance and calibration require outcomes, which may arrive later.
Service telemetry measures availability, latency, errors, and
saturation. One green layer cannot substitute for another.

Population Stability Index (PSI) compares binned reference and current
proportions. Its value depends on binning and sample size and is a
triage signal, not a diagnosis or universal threshold. Governance adds
owners, review cadence, acceptance gates, rollback evidence, privacy,
and an incident path.

### Read the API before running it

- **fixed reference bins:** keep comparison semantics stable; recomputing bins on current data changes the question.
- **`(actual - expected) * log(actual / expected)`:** accumulates PSI contributions after a declared zero-count smoothing policy.
- **monitor → investigate → decide:** separates automated signal detection from causal diagnosis and authorized action.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — calculate PSI with fixed reference quantile bins

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Reference quantiles, smoothing, population, and sample window are documented and remain comparable.

In [ ]:
import numpy as np

reference = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
current = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=float)
edges = np.quantile(reference, [0, 0.25, 0.5, 0.75, 1.0])
edges[0], edges[-1] = -np.inf, np.inf
expected, _ = np.histogram(reference, bins=edges)
actual, _ = np.histogram(current, bins=edges)
expected = np.clip(expected / expected.sum(), 1e-6, None)
actual = np.clip(actual / actual.sum(), 1e-6, None)
psi = np.sum((actual - expected) * np.log(actual / expected))
print({"edges": edges.tolist(), "psi": float(psi)})
assert psi >= 0

**Expected observation:** The shifted current sample produces a positive PSI under bins fixed from the reference sample.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — keep label-available and label-delayed evidence separate

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Available outcomes are representative enough for the scoped estimate; otherwise the metric may be selection-biased.

In [ ]:
monitoring_snapshot = {
    "requests": 10_000,
    "valid_input_rate": 0.998,
    "score_mean": 0.44,
    "outcomes_available": 1_200,
    "labeled_roc_auc": 0.86,
}
label_coverage = (
    monitoring_snapshot["outcomes_available"] / monitoring_snapshot["requests"]
)
print({"label_coverage": label_coverage,
       "performance_estimate": monitoring_snapshot["labeled_roc_auc"]})
assert label_coverage == 0.12

**Expected observation:** The performance metric covers only 12% of requests, so label selection and delay must accompany the score.

### Debugging and practice ramp

**Common mistake:** Treating a PSI threshold as automatic proof of performance degradation or retraining need.

**Diagnostic:** Break the alert into data window, reference, bins, counts, slices, label coverage, service changes, and upstream incidents; reproduce the statistic independently.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define monitoring layers, drift signals, delayed labels, and governance decisions in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not retrain or roll back from one unexplained alert without owner, evidence gate, and impact assessment.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Governance checklist
- Register models in a Model Registry (MLflow, SageMaker, etc.).
- Record: training data snapshot/hash, code version (git sha), hyperparams, metrics.
- Approval workflow for promotion to production (staging → prod).
- Periodic re-validation; drift + performance alerts.

## Learner exercises and progressive hints

1. Compute PSI for multiple features or score windows over time.

**Verify:** Practice 1 — monitoring layers, drift signals, delayed labels, and governance decisions — for each named feature/window, print reference/current counts, fixed bin edges, PSI contributions, and total; assert bins/reference remain frozen and independently recompute one total within 1e-12.

2. Build a small pandas/Matplotlib dashboard of weekly AUC and PSI.

**Verify:** Practice 2 — monitoring layers, drift signals, delayed labels, and governance decisions — save a dashboard with weekly AUC and PSI on labeled separate axes, and print the underlying week/support/AUC/PSI table; mark missing-label weeks rather than silently treating them as zero.

3. Draft a governance policy covering roles, approvals, alerts, and rollback.

**Verify:** Practice 3 — monitoring layers, drift signals, delayed labels, and governance decisions — produce a policy table naming model owner, approver, metric/window/threshold, minimum support, alert route, investigation SLA, rollback trigger, artifact ID, and audit evidence; walk one breached alert through named owners and actions.

### Progressive hints

1. Freeze each feature's reference bin edges and record sample counts beside PSI.
   Test “no shift,” mean shift, and variance shift.
2. Use one row per week with observation count, label coverage, AUC, PSI, and
   model version. Mark missing/delayed labels instead of filling fake metrics.
3. For every threshold, name an owner, review clock, evidence source, action,
   escalation path, and recovery test.

### Additional mastery practice

Design monitoring around observable data, delayed truth, action thresholds, ownership, and rehearsed recovery—not dashboards that merely display numbers.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Label-delay analysis:** Simulate labels arriving 14–30 days after predictions. Build separate views for immediate input/score health and matured performance cohorts.
   **Progressive hint:** Join outcomes by stable prediction ID and evaluate only cohorts whose label window has matured; report label coverage and censoring.

**Verify:** Label-delay analysis — print prediction week, label-available date, immediate-health metrics/support, and matured-cohort AUC/support; assert immature weeks show unavailable performance rather than zero or forward-filled values.

5. **Alert hysteresis:** Design warning and critical thresholds that require persistence or multiple windows, then show how hysteresis prevents alert flapping.
   **Progressive hint:** Use different enter and clear conditions, minimum support, and a cooldown. Preserve raw measurements for audit.

**Verify:** Alert hysteresis — feed a declared metric sequence around warning/critical thresholds, print alert state by window, and assert persistence opens an alert while the lower recovery threshold prevents one-window flapping.

6. **Rollback drill:** Write and rehearse a rollback from model version B to A, including trigger, authority, artifact verification, traffic switch, smoke test, communication, and post-incident evidence.
   **Progressive hint:** A rollback is complete only when the prior artifact, schema, and dependencies remain loadable and the recovery check passes.

**Verify:** Rollback drill — record version-B trigger, authorized actor, verified version-A hash, traffic switch, health/predict smoke results, communication timestamp, and final state; inject a bad rollback artifact and assert the switch stops.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Label-delay analysis


# Practice 5 — Alert hysteresis


# Practice 6 — Rollback drill
